In [1]:
# Financial NER Fine-tuning for FinDoc Validator
# Focus: Bank Statements & Tax Forms Entity Extraction
# Goal: Learn fine-tuning while building a production-ready model

import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

print("🚀 FINANCIAL NER FINE-TUNING BOOTCAMP")
print("="*50)
print("Target: Bank Statements & Tax Forms")
print("Model: DistilBERT (fast, efficient for production)")
print("Dataset: FiNER-ORD + Custom Extensions")

🚀 FINANCIAL NER FINE-TUNING BOOTCAMP
Target: Bank Statements & Tax Forms
Model: DistilBERT (fast, efficient for production)
Dataset: FiNER-ORD + Custom Extensions


In [2]:
# ==========================================
# STEP 1: LOAD AND UNDERSTAND THE DATA
# ==========================================

print("\n📥 Loading FiNER-ORD dataset...")
dataset = load_dataset("gtfintechlab/finer-ord")

# Label mapping from dataset documentation
id2label = {
    0: 'O',        # Outside any entity
    1: 'PER_B',    # Person - Beginning 
    2: 'PER_I',    # Person - Inside
    3: 'LOC_B',    # Location - Beginning
    4: 'LOC_I',    # Location - Inside  
    5: 'ORG_B',    # Organization - Beginning
    6: 'ORG_I'     # Organization - Inside
}

label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

print(f"✅ Dataset loaded: {len(dataset['train'])} train, {len(dataset['validation'])} val, {len(dataset['test'])} test")
print(f"📋 Entity types: {list(set(id2label.values()))}")


📥 Loading FiNER-ORD dataset...
✅ Dataset loaded: 80531 train, 10233 val, 25957 test
📋 Entity types: ['LOC_B', 'PER_I', 'ORG_I', 'LOC_I', 'O', 'ORG_B', 'PER_B']


In [3]:
# ==========================================
# STEP 2: EXTEND LABELS FOR FINANCIAL DOCUMENTS
# ==========================================

print("\n🏦 EXTENDING LABELS FOR BANK STATEMENTS & TAX FORMS")
print("="*50)

# Extended label set for your specific use case
extended_id2label = {
    0: 'O',          # Outside
    1: 'PER_B',      # Person - Beginning 
    2: 'PER_I',      # Person - Inside
    3: 'LOC_B',      # Location - Beginning
    4: 'LOC_I',      # Location - Inside  
    5: 'ORG_B',      # Organization - Beginning
    6: 'ORG_I',      # Organization - Inside
    # NEW LABELS FOR FINANCIAL DOCUMENTS:
    7: 'AMOUNT_B',   # Dollar amounts, percentages
    8: 'AMOUNT_I',   
    9: 'DATE_B',     # Transaction dates, filing deadlines
    10: 'DATE_I',
    11: 'ACCOUNT_B', # Account numbers, routing numbers
    12: 'ACCOUNT_I',
    13: 'SSN_B',     # Social Security Numbers
    14: 'SSN_I',
    15: 'FORM_B',    # Form types (1040, W-2, etc.)
    16: 'FORM_I'
}

extended_label2id = {v: k for k, v in extended_id2label.items()}
extended_num_labels = len(extended_id2label)

print(f"🔥 Extended to {extended_num_labels} labels:")
for i, label in extended_id2label.items():
    if i >= 7:  # Only show new labels
        print(f"   • {label}: For financial document validation")

print("""
💡 STRATEGY: 
   1. Start by fine-tuning on existing PER/LOC/ORG entities
   2. Learn the fine-tuning process thoroughly
   3. Later extend with synthetic data for AMOUNT/DATE/ACCOUNT/SSN/FORM
""")


🏦 EXTENDING LABELS FOR BANK STATEMENTS & TAX FORMS
🔥 Extended to 17 labels:
   • AMOUNT_B: For financial document validation
   • AMOUNT_I: For financial document validation
   • DATE_B: For financial document validation
   • DATE_I: For financial document validation
   • ACCOUNT_B: For financial document validation
   • ACCOUNT_I: For financial document validation
   • SSN_B: For financial document validation
   • SSN_I: For financial document validation
   • FORM_B: For financial document validation
   • FORM_I: For financial document validation

💡 STRATEGY: 
   1. Start by fine-tuning on existing PER/LOC/ORG entities
   2. Learn the fine-tuning process thoroughly
   3. Later extend with synthetic data for AMOUNT/DATE/ACCOUNT/SSN/FORM



In [4]:
# ==========================================
# STEP 3: PREPARE THE MODEL AND TOKENIZER
# ==========================================

print("\n🤖 SETTING UP DISTILBERT FOR TOKEN CLASSIFICATION")
print("="*50)

model_name = "distilbert-base-uncased"
print(f"Loading: {model_name}")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name, 
    num_labels=num_labels,  # Start with original labels
    id2label=id2label,
    label2id=label2id
)

print(f"✅ Model loaded: {model.config.num_labels} output labels")
print(f"✅ Tokenizer loaded: vocab size {tokenizer.vocab_size}")

# M3 MPS device setup
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("🚀 Using M3 MPS acceleration!")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("🔥 Using CUDA GPU!")
else:
    device = torch.device("cpu")
    print("🔧 Using CPU")

print(f"Selected device: {device}")


🤖 SETTING UP DISTILBERT FOR TOKEN CLASSIFICATION
Loading: distilbert-base-uncased


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded: 7 output labels
✅ Tokenizer loaded: vocab size 30522
🚀 Using M3 MPS acceleration!
Selected device: mps


In [5]:
# ==========================================
# STEP 4: DATA PREPROCESSING FOR FINE-TUNING
# ==========================================

print("\n📝 PREPROCESSING DATA FOR FINE-TUNING")
print("="*50)

def tokenize_and_align_labels(examples):
    """
    Critical function: Aligns word labels with subword tokens
    This is where most beginners struggle with NER fine-tuning!
    """
    tokenized_inputs = tokenizer(
        examples["gold_token"], 
        truncation=True, 
        is_split_into_words=True,
        padding=False  # We'll pad later in data collator
    )
    
    labels = []
    for i, label in enumerate(examples["gold_label"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens get -100 (ignored in loss)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the original label
                label_ids.append(label[word_idx])
            else:
                # Other subwords get -100 (ignored) or I-label
                # For simplicity, we'll use -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# ❗ KEY LEARNING MOMENT: Understanding data structure
print("🔍 UNDERSTANDING THE DATA STRUCTURE:")
sample = dataset['train'][:5]
print(f"Sample tokens: {sample['gold_token'][:2]}")
print(f"Sample labels: {sample['gold_label'][:2]}")

# Group tokens by sentences (this is crucial!)
def group_tokens_by_sentence(dataset_split):
    """Group tokens back into sentences for proper tokenization"""
    sentences = []
    labels = []
    
    # Group by doc_idx and sent_idx
    df = dataset_split.to_pandas()
    
    for (doc_idx, sent_idx), group in df.groupby(['doc_idx', 'sent_idx']):
        # Sort by original index to maintain token order
        group = group.sort_index()
        
        sentence_tokens = group['gold_token'].fillna('').tolist()
        sentence_labels = group['gold_label'].tolist()
        
        # Filter out empty tokens
        clean_tokens = []
        clean_labels = []
        for token, label in zip(sentence_tokens, sentence_labels):
            if token and token.strip():
                clean_tokens.append(token.strip())
                clean_labels.append(label)
        
        if clean_tokens:  # Only add non-empty sentences
            sentences.append(clean_tokens)
            labels.append(clean_labels)
    
    return {"gold_token": sentences, "gold_label": labels}

print("\n🔄 Regrouping tokens into sentences...")
train_grouped = group_tokens_by_sentence(dataset['train'])
val_grouped = group_tokens_by_sentence(dataset['validation'])
test_grouped = group_tokens_by_sentence(dataset['test'])

print(f"✅ Grouped into {len(train_grouped['gold_token'])} train sentences")
print(f"✅ Grouped into {len(val_grouped['gold_token'])} validation sentences")
print(f"✅ Grouped into {len(test_grouped['gold_token'])} test sentences")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_dict(train_grouped)
val_dataset = Dataset.from_dict(val_grouped)
test_dataset = Dataset.from_dict(test_grouped)

print("\n🎯 TOKENIZING AND ALIGNING LABELS...")
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)

print("✅ Tokenization complete!")


📝 PREPROCESSING DATA FOR FINE-TUNING
🔍 UNDERSTANDING THE DATA STRUCTURE:
Sample tokens: ['Kenyan', 'Firms']
Sample labels: [0, 0]

🔄 Regrouping tokens into sentences...
✅ Grouped into 3262 train sentences
✅ Grouped into 402 validation sentences
✅ Grouped into 1075 test sentences

🎯 TOKENIZING AND ALIGNING LABELS...


Map:   0%|          | 0/3262 [00:00<?, ? examples/s]

Map:   0%|          | 0/402 [00:00<?, ? examples/s]

Map:   0%|          | 0/1075 [00:00<?, ? examples/s]

✅ Tokenization complete!


In [6]:
# ==========================================
# STEP 5: SETUP TRAINING CONFIGURATION
# ==========================================

print("\n⚙️ CONFIGURING TRAINING PARAMETERS")
print("="*50)

# Data collator for padding
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

# Training arguments - M3 optimized
training_args = TrainingArguments(
    output_dir="../models",
    num_train_epochs=3,              # Start small for learning
    per_device_train_batch_size=32,  # M3 can handle larger batches!
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,                # More frequent logging for M3
    eval_strategy="steps", 
    eval_steps=250,                  # Evaluate more often
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataloader_num_workers=0,        # Important for M3 compatibility!
    report_to=None,                  # Disable wandb for now
)

print(f"🎯 Training for {training_args.num_train_epochs} epochs")
print(f"📊 Batch size: {training_args.per_device_train_batch_size}")
print(f"💾 Model will be saved to: {training_args.output_dir}")


⚙️ CONFIGURING TRAINING PARAMETERS
🎯 Training for 3 epochs
📊 Batch size: 32
💾 Model will be saved to: ../models


In [7]:
# ==========================================
# STEP 6: DEFINE EVALUATION METRICS
# ==========================================

def compute_metrics(eval_pred):
    """Compute accuracy and F1 for each entity type"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (special tokens)
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:  # Ignore special tokens
                true_predictions.append(id2label[pred_id])
                true_labels.append(id2label[label_id])
    
    # Calculate accuracy
    accuracy = accuracy_score(true_labels, true_predictions)
    
    return {
        "accuracy": accuracy,
        "num_predictions": len(true_predictions)
    }


In [8]:
# ==========================================
# STEP 7: CREATE THE TRAINER
# ==========================================

print("\n👨‍🏫 CREATING TRAINER")
print("="*50)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✅ Trainer created successfully!")
print("\n🎓 KEY LEARNING POINTS BEFORE TRAINING:")
print("""
1. **Label Alignment**: We aligned word-level labels with subword tokens
2. **Data Collation**: Automatic padding and batching for efficient training
3. **Evaluation Strategy**: Monitor validation loss to prevent overfitting
4. **Metrics**: Track accuracy across all entity types

🔥 READY TO START FINE-TUNING!
""")



👨‍🏫 CREATING TRAINER
✅ Trainer created successfully!

🎓 KEY LEARNING POINTS BEFORE TRAINING:

1. **Label Alignment**: We aligned word-level labels with subword tokens
2. **Data Collation**: Automatic padding and batching for efficient training
3. **Evaluation Strategy**: Monitor validation loss to prevent overfitting
4. **Metrics**: Track accuracy across all entity types

🔥 READY TO START FINE-TUNING!



In [9]:
# ==========================================
# STEP 8: TRAINING TIME! 
# ==========================================

print("🚀 STARTING FINE-TUNING PROCESS...")
print("="*50)
print("⏰ This will take 5-15 minutes depending on your hardware")
print("📊 Watch the logs to see loss decreasing and accuracy improving!")

trainer.train()

print("""
🎯 TO START TRAINING, UNCOMMENT: trainer.train()

⚡ TRAINING TIPS:
1. Watch for decreasing training/validation loss
2. If validation loss stops improving, training is done
3. GPU will speed this up significantly
4. Expect ~85-90% accuracy on validation set

📈 AFTER TRAINING:
1. Evaluate on test set
2. Test with sample financial sentences
3. Save the model for deployment
4. Plan extensions for AMOUNT/DATE/ACCOUNT entities
""")


🚀 STARTING FINE-TUNING PROCESS...
⏰ This will take 5-15 minutes depending on your hardware
📊 Watch the logs to see loss decreasing and accuracy improving!


Step,Training Loss,Validation Loss,Accuracy,Num Predictions
250,0.048000,0.076674,0.976935,10232



🎯 TO START TRAINING, UNCOMMENT: trainer.train()

⚡ TRAINING TIPS:
1. Watch for decreasing training/validation loss
2. If validation loss stops improving, training is done
3. GPU will speed this up significantly
4. Expect ~85-90% accuracy on validation set

📈 AFTER TRAINING:
1. Evaluate on test set
2. Test with sample financial sentences
3. Save the model for deployment
4. Plan extensions for AMOUNT/DATE/ACCOUNT entities



In [10]:
# ==========================================
# STEP 9: POST-TRAINING EVALUATION CODE
# ==========================================

print("\n📊 POST-TRAINING EVALUATION FUNCTIONS")
print("="*50)

def evaluate_model_detailed(trainer, test_dataset):
    """Detailed evaluation with per-entity metrics"""
    print("🔍 Running detailed evaluation...")
    
    # Get predictions
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=2)
    y_true = predictions.label_ids
    
    # Extract valid predictions (ignore -100 labels)
    true_predictions = []
    true_labels = []
    
    for pred_seq, label_seq in zip(y_pred, y_true):
        for pred, label in zip(pred_seq, label_seq):
            if label != -100:
                true_predictions.append(id2label[pred])
                true_labels.append(id2label[label])
    
    # Print classification report
    print("\n📋 CLASSIFICATION REPORT:")
    print(classification_report(true_labels, true_predictions))
    
    return true_predictions, true_labels

def test_on_sample_sentences(model, tokenizer):
    """Test the trained model on sample financial sentences"""
    print("\n🧪 TESTING ON SAMPLE SENTENCES")
    print("="*40)
    
    # Sample sentences that might appear in bank statements or tax forms
    sample_sentences = [
        "John Smith deposited $5000 into account 123456789 on March 15, 2024.",
        "Microsoft Corporation reported quarterly earnings to the SEC.",
        "The taxpayer's Social Security Number is 123-45-6789.",
        "Bank of America processed the wire transfer on December 1st.",
        "Form 1040 must be filed by April 15th for tax year 2023."
    ]
    
    model.eval()
    with torch.no_grad():
        for sentence in sample_sentences:
            # Tokenize
            tokens = tokenizer(sentence, return_tensors="pt")
            
            # Get predictions
            outputs = model(**tokens)
            predictions = torch.argmax(outputs.logits, dim=2)
            
            # Decode predictions
            word_ids = tokens.word_ids()
            words = sentence.split()
            predicted_labels = []
            
            for word_idx in range(len(words)):
                # Find the first subword for this word
                for i, w_id in enumerate(word_ids):
                    if w_id == word_idx:
                        predicted_labels.append(id2label[predictions[0][i].item()])
                        break
                else:
                    predicted_labels.append('O')
            
            # Print results
            print(f"\nSentence: {sentence}")
            print("Predictions:")
            for word, label in zip(words, predicted_labels):
                if label != 'O':
                    print(f"  🏷️  {word} → {label}")

print("""
🎯 NEXT STEPS AFTER TRAINING:

1. **Run Evaluation**: 
   true_preds, true_labels = evaluate_model_detailed(trainer, tokenized_test)

2. **Test Sample Sentences**:
   test_on_sample_sentences(model, tokenizer)

3. **Save Your Model**:
   model.save_pretrained("./financial-ner-model")
   tokenizer.save_pretrained("./financial-ner-model")

4. **Extend for Your Use Case**:
   - Add synthetic training data for AMOUNT, DATE, ACCOUNT entities
   - Create validation rules for bank statements vs tax forms
   - Integrate with your AWS Step Functions pipeline
""")

print("\n" + "="*60)
print("🎓 CONGRATULATIONS! You're learning production NER fine-tuning!")
print("🚀 This model will be the core of your FinDoc Validator system!")
print("="*60)


📊 POST-TRAINING EVALUATION FUNCTIONS

🎯 NEXT STEPS AFTER TRAINING:

1. **Run Evaluation**: 
   true_preds, true_labels = evaluate_model_detailed(trainer, tokenized_test)

2. **Test Sample Sentences**:
   test_on_sample_sentences(model, tokenizer)

3. **Save Your Model**:
   model.save_pretrained("./financial-ner-model")
   tokenizer.save_pretrained("./financial-ner-model")

4. **Extend for Your Use Case**:
   - Add synthetic training data for AMOUNT, DATE, ACCOUNT entities
   - Create validation rules for bank statements vs tax forms
   - Integrate with your AWS Step Functions pipeline


🎓 CONGRATULATIONS! You're learning production NER fine-tuning!
🚀 This model will be the core of your FinDoc Validator system!


In [11]:
# Get detailed metrics 
print("🔍 RUNNING DETAILED EVALUATION...")
true_preds, true_labels = evaluate_model_detailed(trainer, tokenized_test)

🔍 RUNNING DETAILED EVALUATION...
🔍 Running detailed evaluation...



📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

       LOC_B       0.80      0.89      0.84       300
       LOC_I       0.84      0.61      0.71       128
           O       0.99      0.99      0.99     24112
       ORG_B       0.79      0.84      0.82       544
       ORG_I       0.84      0.75      0.80       389
       PER_B       0.96      0.91      0.93       284
       PER_I       0.99      0.93      0.96       182

    accuracy                           0.98     25939
   macro avg       0.89      0.85      0.86     25939
weighted avg       0.98      0.98      0.98     25939



In [12]:
# Test your model on financial sentences!
print("🧪 TESTING ON FINANCIAL SENTENCES...")

# Patch: Move tensors to the correct device for MPS!
def test_on_sample_sentences_with_device(model, tokenizer, device):
	print("\n🧪 TESTING ON SAMPLE SENTENCES")
	print("="*40)
	sample_sentences = [
		"John Smith deposited $5000 into account 123456789 on March 15, 2024.",
		"Microsoft Corporation reported quarterly earnings to the SEC.",
		"The taxpayer's Social Security Number is 123-45-6789.",
		"Bank of America processed the wire transfer on December 1st.",
		"Form 1040 must be filed by April 15th for tax year 2023."
	]
	model.eval()
	with torch.no_grad():
		for sentence in sample_sentences:
			tokens = tokenizer(sentence, return_tensors="pt")
			# Move tensors to the correct device
			tokens = {k: v.to(device) for k, v in tokens.items()}
			outputs = model(**tokens)
			predictions = torch.argmax(outputs.logits, dim=2)
			word_ids = tokens['input_ids'].cpu()  # for tokenizer.word_ids()
			# Use tokenizer's fast method to get word_ids
			word_id_list = tokenizer(sentence, return_offsets_mapping=True, return_tensors="pt").word_ids()
			words = sentence.split()
			predicted_labels = []
			for word_idx in range(len(words)):
				for i, w_id in enumerate(word_id_list):
					if w_id == word_idx:
						predicted_labels.append(id2label[predictions[0][i].item()])
						break
				else:
					predicted_labels.append('O')
			print(f"\nSentence: {sentence}")
			print("Predictions:")
			for word, label in zip(words, predicted_labels):
				if label != 'O':
					print(f"  🏷️  {word} → {label}")

test_on_sample_sentences_with_device(model, tokenizer, device)

🧪 TESTING ON FINANCIAL SENTENCES...

🧪 TESTING ON SAMPLE SENTENCES

Sentence: John Smith deposited $5000 into account 123456789 on March 15, 2024.
Predictions:
  🏷️  John → PER_B
  🏷️  Smith → PER_I

Sentence: Microsoft Corporation reported quarterly earnings to the SEC.
Predictions:
  🏷️  Microsoft → ORG_B
  🏷️  Corporation → ORG_I
  🏷️  SEC. → ORG_B

Sentence: The taxpayer's Social Security Number is 123-45-6789.
Predictions:

Sentence: Bank of America processed the wire transfer on December 1st.
Predictions:
  🏷️  Bank → ORG_B
  🏷️  of → ORG_I
  🏷️  America → ORG_I

Sentence: Form 1040 must be filed by April 15th for tax year 2023.
Predictions:


In [13]:
# Save for later use in your AWS pipeline!
print("💾 SAVING YOUR TRAINED MODEL...")
model.save_pretrained("./models/financial-ner-v1")
tokenizer.save_pretrained("./models/financial-ner-v1")
print("✅ Model saved successfully!")

💾 SAVING YOUR TRAINED MODEL...
✅ Model saved successfully!


In [14]:
# Document validation function
def validate_document_fields(extracted_entities, document_type):
    """
    Check if required fields are present for different document types
    """
    required_fields = {
        'bank_statement': ['ORG', 'ACCOUNT', 'DATE'],  # Bank name, account info, dates
        'tax_form': ['PER', 'SSN', 'AMOUNT'],          # Person name, SSN, income amounts
        'loan_application': ['PER', 'ORG', 'AMOUNT']   # Applicant, employer, income
    }
    
    missing_fields = []
    for field_type in required_fields[document_type]:
        if not any(entity.startswith(field_type) for entity in extracted_entities):
            missing_fields.append(field_type)
    
    return missing_fields